In [1]:
import pandas as pd
import altair as alt
from pathlib import Path

In [4]:
file_path = Path("../data/education/INEGI_SIESVIM_Cuadro_618.xlsx")

df_raw = pd.read_excel(file_path, sheet_name=0, header=None)

display(df_raw.head(20))
print(df_raw.shape)

,0,1,2,3,4,5,6,7,8
0,PORCENTAJE DE LA POBLACIÓN DE 5 Y MÁS AÑOS SEG...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Año,Sexo,Estados Unidos Mexicanos,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Ninguno,Preescolar o kinder,Primaria,Secundaria,"Preparatoria, bachillerato o equivalente",Educación superior,No especificado
4,1990,Total,13,3.6,48.4,16.8,9.6,6.3,2.3
5,1990,Mujeres,14.3,3.5,48.5,16.2,9.8,5.4,2.4
6,1990,Hombres,11.7,3.6,48.3,17.5,9.3,7.4,2.2
7,1995,Total,9.8,4.3,47.6,22.4,7.8,7.7,0.4
8,1995,Mujeres,10.9,4,48.4,22.8,6.9,6.6,0.3
9,1995,Hombres,8.7,4.5,46.7,22,8.7,8.9,0.5


(28, 9)


In [5]:
#Cleaning the data frame
education = df_raw.iloc[4:, [0, 1, 7]].copy()

education.columns = ["year", "sex", "higher_education"]

education = education[education["sex"].isin(["Mujeres", "Hombres"])]

education["year"] = pd.to_numeric(education["year"])
education["higher_education"] = pd.to_numeric(
    education["higher_education"], errors="coerce"
)

education["sex"] = education["sex"].replace({
    "Mujeres": "Women",
    "Hombres": "Men"
})

education = education.dropna().sort_values(["year", "sex"])

display(education)

,year,sex,higher_education
6,1990,Men,7.4
5,1990,Women,5.4
9,1995,Men,8.9
8,1995,Women,6.6
12,2000,Men,9.4
11,2000,Women,7.5
15,2005,Men,11.3
14,2005,Women,9.9
18,2010,Men,13.4
17,2010,Women,12.9


In [20]:
colors = alt.Scale(
    domain=["Women", "Men"],
    range=["#8624F5", "#1FC3AA"]
)

base = alt.Chart(education).encode(
    x=alt.X(
        "year:O",
        title="Year",
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "higher_education:Q",
        title="Population aged 5+ with higher education (%)",
        scale=alt.Scale(domain=[4, 18])
    ),
    color=alt.Color("sex:N", scale=colors, legend=None)
)

lines = base.mark_line(strokeWidth=3, point=True)

labels_men = (
    base
    .transform_filter(
        (alt.datum.year == 2020) & (alt.datum.sex == "Men")
    )
    .mark_text(
        align="left",
        baseline="bottom",
        dx=12,
        dy=-8,
        fontSize=13,
        fontWeight="bold"
    )
    .encode(
        text=alt.Text("higher_education:Q", format=".1f")
    )
)

labels_women = (
    base
    .transform_filter(
        (alt.datum.year == 2020) & (alt.datum.sex == "Women")
    )
    .mark_text(
        align="left",
        baseline="top",
        dx=12,
        dy=8,
        fontSize=13,
        fontWeight="bold"
    )
    .encode(
        text=alt.Text("higher_education:Q", format=".1f")
    )
)

chart = (
    (lines + labels_men + labels_women)
    .properties(
        width=650,
        height=350,
        title={
            "text": "Population with higher education in Mexico",
            "subtitle": [
                "Share of the population aged 5 and older with higher education, by sex"
            ]
        }
    )
    .configure_view(stroke=None)
)

chart

alt.LayerChart(...)

In [12]:
chart.save("../images/higher_education_mexico.png", scale_factor=2)

## Now let's create the graph of avarage years of education

In [13]:
file_2025 = "../data/eic2025_genero.xlsx"

df_2025_raw = pd.read_excel(
    file_2025,
    sheet_name="11",
    header=None
)

print(df_2025_raw.shape)
display(df_2025_raw.head(12))

(1010, 34)


,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,INEGI. Encuesta Intercensal 2025. Tabulados co...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Índice
1,Fecha de elaboración: 22/09/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Estimadores de los indicadores con enfoque de ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Género 11
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Entidad federativa,Grupos de edad,Estimador,Población de 15 años y más,NaN,NaN,Porcentaje de población alfabeta,NaN,NaN,NaN,...,NaN,NaN,Porcentaje de población con educación superior4,NaN,NaN,NaN,Grado promedio de escolaridad,NaN,NaN,NaN
6,NaN,NaN,NaN,Total,Mujeres,Hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres,...,Hombres,Diferencia mujeres-hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Estados Unidos Mexicanos,Total,Valor,102249879,53789084,48460795,96.098229,95.528492,96.730609,-1.202116,...,26.506984,-1.268287,24.504469,24.534404,24.471243,0.06316,10.265,10.19451,10.343221,-0.148711
9,Estados Unidos Mexicanos,Total,Error estándar,400094.642785,209878.450707,192879.670116,0.016366,0.019768,0.015707,0.025248,...,0.056097,0.073171,0.123558,0.120853,0.131165,0.178352,0.011964,0.011793,0.012533,0.017209


In [14]:
display(df_2025_raw.iloc[5:10, -8:])

,26,27,28,29,30,31,32,33
5,Porcentaje de población con educación superior4,NaN,NaN,NaN,Grado promedio de escolaridad,NaN,NaN,NaN
6,Total,Mujeres,Hombres,Diferencia mujeres-hombres,Total,Mujeres,Hombres,Diferencia mujeres-hombres
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,24.504469,24.534404,24.471243,0.06316,10.265,10.19451,10.343221,-0.148711
9,0.123558,0.120853,0.131165,0.178352,0.011964,0.011793,0.012533,0.017209


In [ ]:
education_2025 = df_2025_raw.loc[
    (df_2025_raw[0] == "Estados Unidos Mexicanos") &
    (df_2025_raw[2] == "Valor") &
    (df_2025_raw[1] != "Total"),
    [1, 31, 32]
].copy()

education_2025.columns = ["age_group", "Women", "Men"]



education_2025[["Women", "Men"]] = education_2025[
    ["Women", "Men"]
].apply(pd.to_numeric, errors="coerce")

display(education_2025)

,age_group,Women,Men
13,15-29 años,11.439543,11.085867
18,30-44 años,11.416636,11.350708
23,45-59 años,9.767944,9.962349
28,60-74 años,7.784862,8.657557
33,75 años y más,5.098883,5.890380


In [36]:
education_2025_long = education_2025.melt(
    id_vars="age_group",
    var_name="sex",
    value_name="schooling_years"
)

# Negative values place men's bars on the left
education_2025_long["plot_value"] = education_2025_long.apply(
    lambda row: -row["schooling_years"]
    if row["sex"] == "Men"
    else row["schooling_years"],
    axis=1
)

education_2025_long["age_group"] = education_2025_long["age_group"].replace({
    "15-29 años": "15-29 years",
    "30-44 años": "30-44 years",
    "45-59 años": "45-59 years",
    "60-74 años": "60-74 years",
    "75 años y más": "75 years and older"
})

display(education_2025_long)

,age_group,sex,schooling_years,plot_value
0,15-29 years,Women,11.439543,11.439543
1,30-44 years,Women,11.416636,11.416636
2,45-59 years,Women,9.767944,9.767944
3,60-74 years,Women,7.784862,7.784862
4,75 years and older,Women,5.098883,5.098883
5,15-29 years,Men,11.085867,-11.085867
6,30-44 years,Men,11.350708,-11.350708
7,45-59 years,Men,9.962349,-9.962349
8,60-74 years,Men,8.657557,-8.657557
9,75 years and older,Men,5.890380,-5.890380


In [37]:
age_order = [
    "15-29 years",
    "30-44 years",
    "45-59 years",
    "60-74 years",
    "75 years and more"
]

chart_education_2025 = (
    alt.Chart(education_2025_long)
    .mark_bar(size=35)
    .encode(
        y=alt.Y(
            "age_group:N",
            sort=age_order,
            title=None,
            axis=alt.Axis(labelFontSize=12)
        ),
        x=alt.X(
            "plot_value:Q",
            title="Average years of schooling",
            scale=alt.Scale(domain=[-15, 15]),
            axis=alt.Axis(
                labelExpr="abs(datum.value)",
                values=[-15, -10, -5, 0, 5, 10, 15],
                grid=False
            )
        ),
        color=alt.Color(
            "sex:N",
            scale=alt.Scale(
                domain=["Men", "Women"],
                range=["#1FC3AA", "#8624F5"]
            ),
            legend=alt.Legend(
                title=None,
                orient="bottom-right",
                direction = "horizontal",
            )
        ),
        tooltip=[
            alt.Tooltip("age_group:N", title="Age group"),
            alt.Tooltip("sex:N", title="Sex"),
            alt.Tooltip(
                "schooling_years:Q",
                title="Average years",
                format=".1f"
            )
        ]
    )
    .properties(
        width=650,
        height=320,
        title=alt.TitleParams(
            text="Average years of schooling by age and sex",
            subtitle="Mexico, 2025"
        )
    )
)

bars = chart_education_2025

labels_men = (
    alt.Chart(education_2025_long)
    .transform_filter(alt.datum.sex == "Men")
    .mark_text(
        align="right",
        dx=-8,
        fontSize=12,
        fontWeight="bold"
    )
    .encode(
        x="plot_value:Q",
        y=alt.Y("age_group:N", sort=age_order),
        text=alt.Text("schooling_years:Q", format=".1f")
    )
)

labels_women = (
    alt.Chart(education_2025_long)
    .transform_filter(alt.datum.sex == "Women")
    .mark_text(
        align="left",
        dx=8,
        fontSize=12,
        fontWeight="bold"
    )
    .encode(
        x="plot_value:Q",
        y=alt.Y("age_group:N", sort=age_order),
        text=alt.Text("schooling_years:Q", format=".1f")
    )
)

chart_education_2025 = (
    (bars + labels_men + labels_women)
    .configure_view(stroke=None)
)

chart_education_2025

alt.LayerChart(...)

In [39]:
chart_education_2025.save("../images/avarage_years_of_schooling.png", scale_factor=2)
